In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import find_dotenv
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

ROOT = Path(find_dotenv()).parent
df = pd.read_csv(ROOT / "data" / "processed" / "pm25_hourly_clean.csv")
df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Asia/Kolkata")

HORIZON = 24
LAGS = [1, 2, 3, 6, 12, 24, 48]

def mae(a, b):
    return np.mean(np.abs(a - b))

def rmse(a, b):
    return np.sqrt(np.mean((a - b) ** 2))

results = []
importances = {}

for station, g in df.groupby("station"):
    s = g.sort_values("time").set_index("time")["pm25"]

    X = pd.DataFrame({"now": s})
    for k in LAGS:
        X[f"lag_{k}"] = s.shift(k)
    X["roll_mean_24"] = s.rolling(24).mean()
    X["roll_std_24"] = s.rolling(24).std()
    X["hour"] = X.index.hour
    X["dayofweek"] = X.index.dayofweek
    X["month"] = X.index.month
    X["target"] = s.shift(-HORIZON)

    data = X.dropna()
    if len(data) < 500:
        print("Skipping", station, "- only", len(data), "usable rows")
        continue

    split = int(len(data) * 0.8)
    train, test = data.iloc[:split], data.iloc[split:]
    features = [c for c in data.columns if c != "target"]

    rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                               n_jobs=-1, random_state=42)
    rf.fit(train[features], train["target"])

    xgb = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                       subsample=0.8, random_state=42)
    xgb.fit(train[features], train["target"])

    y = test["target"]
    p_base = test["now"]
    p_rf = rf.predict(test[features])
    p_xgb = xgb.predict(test[features])

    results.append({
        "station": station,
        "train_hours": len(train),
        "test_hours": len(test),
        "baseline_MAE": round(mae(y, p_base), 1),
        "RF_MAE": round(mae(y, p_rf), 1),
        "XGB_MAE": round(mae(y, p_xgb), 1),
        "baseline_RMSE": round(rmse(y, p_base), 1),
        "RF_RMSE": round(rmse(y, p_rf), 1),
        "XGB_RMSE": round(rmse(y, p_xgb), 1),
    })
    importances[station] = pd.Series(xgb.feature_importances_, index=features) \
                             .sort_values(ascending=False)

table = pd.DataFrame(results)
print(table.to_string(index=False))
first = next(iter(importances))
print("\nMost important features for", first)
print(importances[first].head(6).round(3))

                      station  train_hours  test_hours  baseline_MAE  RF_MAE  XGB_MAE  baseline_RMSE  RF_RMSE  XGB_RMSE
Anand Vihar, New Delhi - DPCC         4936        1234          30.2    34.8     38.5           40.6     45.7      52.2
                    New Delhi         5310        1328          14.6    15.1     14.6           19.5     18.8      18.4
   Punjabi Bagh, Delhi - DPCC         5228        1308          23.5    84.0     27.7           31.4    104.5      40.6
      R K Puram, Delhi - DPCC         5068        1268          20.5    34.4     24.3           27.4     43.4      34.0

Most important features for Anand Vihar, New Delhi - DPCC
month           0.378
now             0.253
roll_mean_24    0.055
lag_1           0.045
dayofweek       0.040
lag_24          0.038
dtype: float32


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import find_dotenv
from xgboost import XGBRegressor

ROOT = Path(find_dotenv()).parent
df = pd.read_csv(ROOT / "data" / "processed" / "pm25_hourly_clean.csv")
df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Asia/Kolkata")

HORIZON = 24
LAGS = [1, 2, 3, 6, 12, 24, 48]
STATION = "Anand Vihar, New Delhi - DPCC"   # change this to try other stations

g = df[df["station"] == STATION]
s = g.sort_values("time").set_index("time")["pm25"]

X = pd.DataFrame({"now": s})
for k in LAGS:
    X[f"lag_{k}"] = s.shift(k)
X["change_1h"] = s - s.shift(1)
X["change_24h"] = s - s.shift(24)
X["roll_mean_24"] = s.rolling(24).mean()
X["roll_std_24"] = s.rolling(24).std()
X["hour"] = X.index.hour
X["target"] = s.shift(-HORIZON)

data = X.dropna()
split = int(len(data) * 0.8)
train, test = data.iloc[:split], data.iloc[split:]
features = [c for c in data.columns if c != "target"]

model = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=3,
                     subsample=0.8, random_state=42)
model.fit(train[features], train["target"] - train["now"])

pred = np.clip(test["now"] + model.predict(test[features]), 0, None)

# plot a 2-week window from the test period so the lines are readable
window = test.index[-14*24:]
plt.figure(figsize=(12, 4))
plt.plot(window, test.loc[window, "target"], label="Actual PM2.5", color="#12324F")
plt.plot(window, pred.loc[window], label="Forecast (24h ahead)", color="#1B8FCF", linestyle="--")
plt.plot(window, test.loc[window, "now"], label="Baseline (same as now)", color="#A6A6A6", alpha=0.6)
plt.title(f"Predicted vs actual PM2.5, 24h ahead — {STATION.split(',')[0]}")
plt.ylabel("PM2.5 (µg/m³)")
plt.legend()
plt.tight_layout()
plt.savefig(ROOT / "data" / "processed" / "predicted_vs_actual.png", dpi=150)
plt.show()